In [1]:
# Program to show how LLM can call a tool based on the requirement
from langchain.chat_models import init_chat_model
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, ToolMessage, SystemMessage, AIMessage
import requests

In [2]:
@tool
def get_weather(latitude: float, longitude: float) -> dict:
    """function to return weather information based on latitude and longitude"""

    """This is a publically available API that returns the weather for a given location."""
    response = requests.get(
        f"https://api.open-meteo.com/v1/forecast?latitude={latitude}&longitude={longitude}&current_weather=true"
    )
    data = response.json()
    return data["current_weather"]


In [3]:
# gemini-3.5-flash has geo capability 
gemini = init_chat_model("gemini-3.5-flash", model_provider="google_genai")
gemini_with_tools = gemini.bind_tools([get_weather])

In [4]:
system_message = SystemMessage(
    "You are a helpful weather assistant.If needed, find out latitude and logitude then provide to tool. ")

user_message = HumanMessage("What is the weather in Perth")
# user_message = HumanMessage("What is the capital of Australia")

messages = [system_message, user_message]

In [5]:
while True:
    response = gemini_with_tools.invoke(messages)
    messages.append(response)

    if len(response.tool_calls) == 0:
        break

    for tool_call in response.tool_calls:
        func_name = tool_call["name"]
        chosen_function = eval(func_name)

        # Call the tool and add result
        tool_result = chosen_function.invoke(tool_call)
        messages.append(tool_result)    


In [6]:
for message in messages:
    message.pretty_print()

================================ System Message ================================

You are a helpful weather assistant.If needed, find out latitude and logitude then provide to tool. 
================================ Human Message =================================

What is the weather in Perth
================================== Ai Message ==================================

[]
Tool Calls:
  get_weather (call_619149)
 Call ID: call_619149
  Args:
    latitude: -31.9505
    longitude: 115.8605
================================= Tool Message =================================
Name: get_weather

{"time": "2026-10-14T02:30", "interval": 900, "temperature": 25.8, "windspeed": 20.5, "winddirection": 48, "is_day": 1, "weathercode": 0}
================================== Ai Message ==================================

[{'type': 'text', 'text': 'The current weather in Perth is sunny with a temperature of 25.8°C (78.4°F). There is a wind blowing at around 20.5 km/h.', 'extras': {'signature': 'EsADCr0D